# Seven-gene pull — no filtering except the gene list

Builds the same gene-level dataframe (same columns, same source files) as
`ClinVar_wrangle_uniprot_gencc_merge.ipynb`, but for only these seven genes:

`SORL1, TMEM175, RAB32, SH3GL2, RIT2, SV2C, UNC13A`

These are the genes from the neurodegeneration list in `20260911_for_CZI.ipynb` that
fall out of the MAVEable set, so every biological filter in the main pipeline is
**skipped** here and the gene list is the only row filter.

**Filters deliberately dropped** (relative to the main notebook):
- `Type == 'single nucleotide variant'` — indels/deletions/etc. are kept
- `ClinicalSignificance` must contain pathogenic/benign/uncertain, and must not contain
  penetrance/drug/other/protective/risk — all classifications are kept (so `-` and
  `risk factor` show up as extra count columns)
- `ReviewStatus` 1-star minimum (`no assertion` removal) — all review statuses are kept
- missense-only (`p.variant` not ending in `*` or `=`, and non-empty) — all consequences kept
- inner merge against GenCC moderate-and-above — none of these seven genes are in GenCC,
  which is what excluded them in the first place, so the merge is dropped entirely
- `Length < 592` from `20260911_for_CZI.ipynb`

**One filter kept:** `Assembly == 'GRCh38'`. This is a project rule (CLAUDE.md), and it is
not a biological filter — ClinVar ships one row per assembly, so without it every variant
is counted twice.

All the annotation merges (HAP1, UniProt, Lacoste, Rendo toxicity, VAMP-seq-ome, OpenCell,
Protein Atlas localization, T cell screen) are left joins in the main pipeline and are kept
as-is, so they never drop rows.

In [1]:
import glob
import re

import pandas as pd
import numpy as np

In [2]:
# The seven genes that dropped out of the MAVEable set
GENES = ['SORL1', 'TMEM175', 'RAB32', 'SH3GL2', 'RIT2', 'SV2C', 'UNC13A']

## 1. ClinVar — filter to the seven genes only

Read in chunks and keep only the seven genes, so the full ~440 MB table never has to sit in
memory at once. No `Type`, `ClinicalSignificance` or `ReviewStatus` filtering.

In [3]:
# Locate the ClinVar dump (not committed to the repo; download from the NCBI FTP site)
cv_files = sorted(glob.glob('variant_summary*.txt.gz'))
if not cv_files:
    raise FileNotFoundError('No variant_summary_*.txt.gz file found in the working directory.')
cv_file = cv_files[-1]          # most recent if several copies exist
print(f'Loading: {cv_file}')

chunks = []
multi_gene_rows = 0
for chunk in pd.read_csv(cv_file, sep='\t', low_memory=False, chunksize=500_000):
    hit = chunk[chunk['GeneSymbol'].isin(GENES)]
    chunks.append(hit)
    # Rows where one of our genes appears inside a multi-gene GeneSymbol string
    # (e.g. 'SORL1;LOC...'). These are NOT kept: the whole pipeline groups by GeneSymbol,
    # so a combined symbol would become its own spurious "gene".
    multi_gene_rows += chunk['GeneSymbol'].str.contains('|'.join(GENES), na=False).sum() - len(hit)

ClinVar_seven = pd.concat(chunks, ignore_index=True)
print(f'Rows for the seven genes (all assemblies): {len(ClinVar_seven):,}')
print(f'Multi-gene GeneSymbol rows excluded:       {multi_gene_rows:,}')

# GRCh38 only — ClinVar carries one row per assembly, so this is de-duplication, not a filter
ClinVar_seven = ClinVar_seven[ClinVar_seven['Assembly'] == 'GRCh38']
print(f'Rows on GRCh38:                            {len(ClinVar_seven):,}')

print()
print(ClinVar_seven.groupby('GeneSymbol').size().to_string())
print()
print('Variant types kept (the main pipeline would keep only SNVs):')
print(ClinVar_seven['Type'].value_counts().to_string())
print()
print('ClinicalSignificance values kept (the main pipeline would drop several of these):')
print(ClinVar_seven['ClinicalSignificance'].value_counts().to_string())

Loading: variant_summary_20260911.txt.gz


Rows for the seven genes (all assemblies): 3,587
Multi-gene GeneSymbol rows excluded:       254
Rows on GRCh38:                            1,787

GeneSymbol
RAB32        30
RIT2         25
SH3GL2       42
SORL1      1012
SV2C        122
TMEM175     128
UNC13A      428

Variant types kept (the main pipeline would keep only SNVs):
Type
single nucleotide variant    1718
Deletion                       33
Microsatellite                 11
Indel                          10
Duplication                     8
Variation                       3
Insertion                       3
Inversion                       1

ClinicalSignificance values kept (the main pipeline would drop several of these):
ClinicalSignificance
Uncertain significance                          1100
Likely benign                                    380
Benign                                           131
-                                                 99
Conflicting classifications of pathogenicity      27
Pathogenic             

In [4]:
# For reference only: none of these seven genes are in the GenCC moderate-and-above list,
# which is the inner merge that excluded them from MAVEable. Nothing is filtered on it here.
unique_gencc = pd.read_csv('unique_gencc-submissions_20260911.csv')
in_gencc = sorted(set(GENES) & set(unique_gencc['gene_symbol']))
print(f'In GenCC (moderate+): {in_gencc if in_gencc else "none"}')
print(f'Not in GenCC:         {sorted(set(GENES) - set(unique_gencc["gene_symbol"]))}')

In GenCC (moderate+): ['RAB32', 'UNC13A']
Not in GenCC:         ['RIT2', 'SH3GL2', 'SORL1', 'SV2C', 'TMEM175']


## 2. Protein change annotation

Same `p.variant` extraction and three-letter → one-letter conversion as the main notebook.
The column is computed for inspection but is **not** used to filter to missense, and it is
dropped from the final output just as it is in the main pipeline.

In [5]:
# Extract the p.variant into a new column, handling cases where p.variant is not present
ClinVar_seven['p.variant'] = ClinVar_seven['Name'].str.extract(r'\(p\.([A-Za-z]{3}\d+[A-Za-z]*?)\)')[0]

amino_acid_map = {
    'Ala': 'A', 'Cys': 'C', 'Asp': 'D', 'Glu': 'E', 'Phe': 'F',
    'Gly': 'G', 'His': 'H', 'Ile': 'I', 'Lys': 'K', 'Leu': 'L',
    'Met': 'M', 'Asn': 'N', 'Pro': 'P', 'Gln': 'Q', 'Arg': 'R',
    'Ser': 'S', 'Thr': 'T', 'Val': 'V', 'Trp': 'W', 'Tyr': 'Y',
    'Ter': '*'  # Special case for stop codon
}

# Function to replace three-letter codes with single-letter codes using regex
def replace_aa_codes(p_variant):
    if isinstance(p_variant, str):  # Check if p_variant is a string
        # Define a regex pattern for matching three-letter amino acid codes
        pattern = r'([A-Z][a-z][a-z])'

        # Replace three-letter codes with single-letter codes
        def replace_match(match):
            three_letter_code = match.group(0)  # Get the matched three-letter code
            return amino_acid_map.get(three_letter_code, three_letter_code)  # Return single-letter or original

        # Use regex sub with the replacement function
        replaced_variant = re.sub(pattern, replace_match, p_variant)
        return replaced_variant

    return ''  # Return empty string for non-matching cases

# Apply the replacement function
ClinVar_seven['p.variant'] = ClinVar_seven['p.variant'].apply(replace_aa_codes)

# NOTE: the main pipeline subsets to missense here. Skipped on purpose.
n_missense = (~ClinVar_seven['p.variant'].str.endswith(('*', '=')) & (ClinVar_seven['p.variant'] != '')).sum()
print(f'Rows kept:                          {len(ClinVar_seven):,}')
print(f'(of which missense, for reference): {n_missense:,}')

Rows kept:                          1,787
(of which missense, for reference): 1,180


## 3. HAP1 essentiality and UniProt length / localization

Both are left joins, identical to the main notebook.

In [6]:
# load hap1 data:
hap1_essential = pd.read_excel('hap1_essential.csv.xlsx')
hap1_essential = hap1_essential.rename(columns={'selected': 'hap1_essential'})

# merge with ClinVar
CV_seven_hap1 = pd.merge(ClinVar_seven, hap1_essential, left_on='GeneSymbol', right_on='Gene', how='left')

In [7]:
uniprot_len_loc = pd.read_csv('uniprot_len_loc_2.csv')

In [8]:
# Create expanded Y dataframe with both Gene_Name and Synonyms
Y_expanded = []
# Add rows for Gene_Name
Y_temp = uniprot_len_loc.copy()
Y_temp['match_column'] = Y_temp['Gene_Name']
Y_expanded.append(Y_temp)
# Add rows for each synonym
Y_synonyms = uniprot_len_loc.copy()
Y_synonyms['Synonyms'] = Y_synonyms['Synonyms'].str.split(',')
Y_synonyms = Y_synonyms.explode('Synonyms')
Y_synonyms['Synonyms'] = Y_synonyms['Synonyms'].str.strip()
Y_synonyms['match_column'] = Y_synonyms['Synonyms']
Y_expanded.append(Y_synonyms)
# Combine all possible matches
Y_all_matches = pd.concat(Y_expanded, ignore_index=True)
# Remove duplicates (prioritize Gene_Name matches)
Y_all_matches = Y_all_matches.drop_duplicates(subset=['match_column'], keep='first')

# Do the merge with suffixes to handle duplicate column names
CV_seven_hap1_loc_len = pd.merge(CV_seven_hap1, Y_all_matches,
                                 left_on='GeneSymbol', right_on='match_column',
                                 how='left', suffixes=('', '_y'))

# Function to clean up redundant columns
def clean_redundant_columns(df):
    """
    Remove redundant columns that are identical between _x and _y versions,
    keeping the original (non-suffixed) version when possible.
    """
    columns_to_drop = []

    # Find all columns that have _y suffix
    y_columns = [col for col in df.columns if col.endswith('_y')]

    for y_col in y_columns:
        base_name = y_col[:-2]  # Remove '_y' suffix

        # Check if there's a corresponding column without suffix
        if base_name in df.columns:
            # Fill NaN values in original with values from _y column
            df[base_name] = df[base_name].fillna(df[y_col])
            columns_to_drop.append(y_col)
        else:
            # If no original column exists, rename _y column to remove suffix
            df[base_name] = df[y_col]
            columns_to_drop.append(y_col)

    # Drop the redundant _y columns
    df = df.drop(columns=columns_to_drop)

    return df

# Clean up the merged dataframe
CV_seven_hap1_loc_len = clean_redundant_columns(CV_seven_hap1_loc_len)

# Optional: Drop the helper match_column if you don't need it
if 'match_column' in CV_seven_hap1_loc_len.columns:
    CV_seven_hap1_loc_len = CV_seven_hap1_loc_len.drop('match_column', axis=1)

print(f'Final dataframe shape: {CV_seven_hap1_loc_len.shape}')
print()
print(CV_seven_hap1_loc_len.groupby('GeneSymbol')[['Entry', 'Length', 'secreted']].first().to_string())

Final dataframe shape: (1787, 65)

             Entry  Length secreted
GeneSymbol                         
RAB32       Q13637     225       no
RIT2        Q99578     217       no
SH3GL2      Q99962     352       no
SORL1       Q92673    2214      yes
SV2C        Q496J9     727       no
TMEM175     Q9BSA9     504       no
UNC13A      Q9UPW8    1703       no


## 4. Localization flags and hierarchical class

In [9]:
CV_seven_hap1_loc_len['cytoplasmic'] = CV_seven_hap1_loc_len['Subcellular location [CC]'].str.contains('cytoplasm|cell membrane|multi-pass|projection|single-pass|reticulum|mito', case=False, na=False).map({True: 'yes', False: 'no'})

In [10]:
CV_seven_hap1_loc_len['nuclear'] = CV_seven_hap1_loc_len['Subcellular location [CC]'].str.contains('nucle', case=False, na=False).map({True: 'yes', False: 'no'})

In [11]:
# Create the localization class column with hierarchical logic
# Priority: hap1_essential (SGE) > secreted > cytoplasmic > nuclear
CV_seven_hap1_loc_len['class'] = np.where(CV_seven_hap1_loc_len['hap1_essential'] == 'YES', 'SGE',
               np.where(CV_seven_hap1_loc_len['secreted'] == 'yes', 'secreted',
                np.where(CV_seven_hap1_loc_len['cytoplasmic'] == 'yes', 'cytoplasmic',
                 np.where(CV_seven_hap1_loc_len['nuclear'] == 'yes', 'nuclear', 'other'))))

## 5. Collapse to one row per gene

Recode of the combined ClinVar classes and the groupby are unchanged from the main notebook.
Because no `ClinicalSignificance` filter was applied, the count columns include categories
the main pipeline never sees (`-`, `risk factor`, and so on).

In [12]:
CV_seven_hap1_loc_len['ClinicalSignificance'] = CV_seven_hap1_loc_len['ClinicalSignificance'].replace({
    'Pathogenic/Likely pathogenic': 'Likely pathogenic',
    'Benign/Likely benign': 'Likely benign'
})

In [13]:
# Specify only the columns that are variant-specific (that we DON'T want to keep as-is)
variant_specific_columns = ['ClinicalSignificance']  # Add any other variant-specific columns here

# Get all other columns EXCEPT GeneSymbol (since it's the groupby key)
gene_level_columns = [col for col in CV_seven_hap1_loc_len.columns
                     if col not in variant_specific_columns and col != 'GeneSymbol']

# For gene-level columns, take the first value for each gene (they should be the same anyway)
gene_info = CV_seven_hap1_loc_len.groupby('GeneSymbol')[gene_level_columns].first().reset_index()

# Get counts for each ClinicalSignificance class per gene
significance_counts = CV_seven_hap1_loc_len.groupby('GeneSymbol')['ClinicalSignificance'].value_counts().unstack(fill_value=0)
significance_counts = significance_counts.reset_index()
significance_counts.columns.name = None

# Merge gene info with significance counts
collapsed_df = pd.merge(gene_info, significance_counts, on='GeneSymbol', how='left')

# Fill any NaN values in count columns with 0
count_columns = [col for col in collapsed_df.columns if col not in gene_level_columns + ['GeneSymbol']]
for col in count_columns:
    collapsed_df[col] = collapsed_df[col].fillna(0).astype(int)

print(f'Original dataframe shape: {CV_seven_hap1_loc_len.shape}')
print(f'Collapsed dataframe shape: {collapsed_df.shape}')
print(f'ClinicalSignificance count columns: {count_columns}')

collapsed_CV_seven = collapsed_df

Original dataframe shape: (1787, 68)
Collapsed dataframe shape: (7, 75)
ClinicalSignificance count columns: ['-', 'Benign', 'Conflicting classifications of pathogenicity', 'Likely benign', 'Likely pathogenic', 'Pathogenic', 'Uncertain significance', 'risk factor']


## 6. Annotation merges

Same files and same left joins as the back half of the main notebook, including the T cell
essentiality screen (Shifrut et al., 2018) that produced the `T_cell` column in
`MAVEable_final_isge.csv`.

In [14]:
# Importing other files
# The Lacoste et al., 2025 pathogenic mislocalization data
lacoste = pd.read_csv('lacoste.csv')
# The Rendo 2025 toxic overexpression list
toxic = pd.read_csv('Rendo_2025_toxic.csv')
# the VAMP-Seqome dataframe previously generated by Shawn
vampseqome = pd.read_csv('VAMPseq-ome.csv')
# Subcellular localization from the Human Protein Atlas
localization = pd.read_csv('subcellular_location.tsv', sep='\t')
# Opencell library data on tagging
opencell = pd.read_csv('opencell-library-metadata.csv')
# T cell essentiality from Shifrut et al., 2018
tcell = pd.read_csv('Tcell_screen.csv')

In [15]:
# manipulate the lacoste dataframe to make a list of genes with mislocalization phenotypes
# Drop rows where Mislocalized is NaN
lacoste_clean = lacoste.dropna(subset=['Mislocalized']).copy()

# Convert to integer
lacoste_clean['Mislocalized'] = lacoste_clean['Mislocalized'].astype(int)

# Group and aggregate
collapsed = (
    lacoste_clean
    .groupby('Gene', as_index=False)
    .agg({
        'Mislocalized': 'max',
        'Mislocalization phenotype': lambda x: '; '.join(sorted(set(x.dropna()))) if any(lacoste_clean.loc[x.index, 'Mislocalized'] == 1) else pd.NA
    }))

# Merge the lacoste classifications with the seven-gene dataframe
maveable_merged = collapsed_CV_seven.merge(
    collapsed[['Gene', 'Mislocalized', 'Mislocalization phenotype']],
    how='left',
    left_on='GeneSymbol',
    right_on='Gene'
)

# Drop the redundant 'Gene' column from collapsed ('Gene' from HAP1 survives as Gene_x)
maveable_merged = maveable_merged.drop(columns=['Gene_y'])

In [16]:
# now add in the toxicity data
maveable_merged['is_Toxic_Gene'] = maveable_merged['GeneSymbol'].isin(toxic['genes'])

# now add in the VAMP-Seqome dataframe from Shawn
# Columns to bring over from vampseqome
cols_to_merge = [
    'Gene', 'Disease', 'B_cell', 'HAP1', 'Known_complex', 'ComplexName',
    'GFP_tagged', 'GFP_score', 'meltPoint', 'Stable', 'Secreted'
]

# Perform the merge
maveable_merged = maveable_merged.merge(
    vampseqome[cols_to_merge],
    how='left',
    left_on='GeneSymbol',
    right_on='Gene'
)

# Drop the extra 'Gene' column from vampseqome if not needed
maveable_merged = maveable_merged.drop(columns=['Gene'])

# Now bring in the CZI gene tagging info
maveable_merged = maveable_merged.merge(
    opencell[['gene_name', 'tagged_terminus']],
    how='left',
    left_on='GeneSymbol',
    right_on='gene_name'
)

# Drop the extra 'gene_name' column if not needed
maveable_merged = maveable_merged.drop(columns=['gene_name'])

# bring in the protein atlas locations
maveable_merged = maveable_merged.merge(
    localization[['Gene name', 'Main location', 'Additional location']],
    how='left',
    left_on='GeneSymbol',
    right_on='Gene name'
)

# add in the t cell data
# drop rows with less than 3 sgRNAs, then require a depleting log fold change
tcell_filtered = tcell[tcell['neg|goodsgrna'] >= 3]
tcell_lfc_filtered = tcell_filtered[tcell_filtered['neg|lfc'] <= -0.2]
tcellessential = tcell_lfc_filtered['id']

maveable_merged['T_cell'] = np.where(
    maveable_merged['GeneSymbol'].isin(tcellessential),
    'Essential',
    'Non-essential')

# Drop the extra 'Gene name' column if not needed
maveable_merged = maveable_merged.drop(columns=['Gene name'])

## 7. Drop the same columns and add `location_feasible`

Same `columns_to_drop` list as the main notebook, minus `gene_symbol` (no GenCC merge here)
and minus the `Gene_x`/`Start`/`Stop` bookkeeping differences, plus the stray `Unnamed:`
index columns that ride along from the CSV inputs.

In [17]:
# Clean up the data frame by getting rid of redundant columns
# Also removing columns that I (Dan, on 20250731) think are irrellevant, e.g. some columns about
# a single SNP that I think are just artifacts of how the dataframes were collasped
columns_to_drop = [
    "Type", "Name", "ClinSigSimple", "LastEvaluated", "RS# (dbSNP)", "nsv/esv (dbVar)",
    "RCVaccession", "ReferenceAllele", "AlternateAllele", "ReviewStatus",
    "NumberSubmitters", "OtherIDs", "Guidelines", "SubmitterCategories", "VariationID",
    "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF", "SomaticClinicalImpact",
    "SomaticClinicalImpactLastEvaluated", "ReviewStatusClinicalImpact", "Oncogenicity",
    "OncogenicityLastEvaluated", "ReviewStatusOncogenicity", "SCVsForAggregateGermlineClassification",
    "SCVsForAggregateOncogenicityClassification", "p.variant", "Gene_x", "ENSEMBL_ID",
    "tot.sense", "tot.anti", "p.val", "q.val", "ratio", "hap1_essential",
    "Reviewed", "Entry Name", "Organism", "Gene_Name",
    "Start", "Stop"
]

maveable_merged = maveable_merged.drop(columns=[c for c in columns_to_drop if c in maveable_merged.columns])

# Unnamed index columns carried in from the source CSVs
maveable_merged = maveable_merged.drop(columns=[c for c in maveable_merged.columns if str(c).startswith('Unnamed:')])

In [18]:
# make a boolean based on the list of locations Dan specified
# Your allowed unique first entries
feasible_locations = ['Centrosome', 'Nucleoplasm', 'Cytosol', 'Nuclear bodies', 'Nucleoli', 'Mitotic chromosome', 'Endoplasmic reticulum', 'Basal body', 'Nucleoli fibrillar center', 'Actin filaments', 'Nuclear membrane', 'Cytoplasmic bodies', 'Midbody ring', 'Centriolar satellite', 'Nuclear speckles', 'Cytokinetic bridge', 'Mitotic spindle', 'Intermediate filaments', 'Microtubules', 'Lipid droplets', 'Aggresome', 'Nucleoli rim']

# First entries from each column
first_additional = maveable_merged['Additional location'].str.split(';').str[0]
first_main       = maveable_merged['Main location'].str.split(';').str[0]

# Boolean: True if either matches feasible_locations
maveable_merged['location_feasible'] = (
    first_additional.isin(feasible_locations) | first_main.isin(feasible_locations)
)

## 8. Match the column order of `MAVEable_final_is_feasible.csv`

The canonical order is read straight off the existing output so the two tables line up.
ClinicalSignificance categories that exist in the filtered pipeline but not in these seven
genes are added as zero columns; categories that only exist here (because no significance
filter was applied) are appended at the end.

In [19]:
# Canonical column order, taken from the existing final output (read-only, header only)
reference_columns = pd.read_csv('MAVEable_final_is_feasible.csv', nrows=0).columns
canonical = [c for c in reference_columns if not str(c).startswith('Unnamed:')]

missing = [c for c in canonical if c not in maveable_merged.columns]
extra   = [c for c in maveable_merged.columns if c not in canonical]

print(f'Count columns present in MAVEable_final_is_feasible.csv but not here (filled with 0):')
for c in missing:
    print(f'  {c}')
print(f'\nColumns present only here (kept, appended at the end):')
for c in extra:
    print(f'  {c}')

# Every missing column is a ClinicalSignificance count category, so 0 is the right fill
for c in missing:
    maveable_merged[c] = 0

seven_genes_df = maveable_merged[canonical + extra]
print(f'\nFinal shape: {seven_genes_df.shape}')

Count columns present in MAVEable_final_is_feasible.csv but not here (filled with 0):
  Benign/Likely benign; association
  Benign; association
  Conflicting classifications of pathogenicity; Affects
  Conflicting classifications of pathogenicity; association
  Likely benign; association
  Likely pathogenic; association
  Pathogenic/Likely pathogenic; association
  Uncertain significance/VUS-high
  Uncertain significance/VUS-low
  Uncertain significance/VUS-mid
  Uncertain significance; Affects
  Uncertain significance; association

Columns present only here (kept, appended at the end):
  -
  risk factor

Final shape: (7, 63)


In [20]:
seven_genes_df[['GeneSymbol', 'Length', 'class', 'Pathogenic', 'Likely pathogenic',
                'Uncertain significance', 'Benign', 'Likely benign',
                'Conflicting classifications of pathogenicity',
                'T_cell', 'HAP1', 'Main location', 'location_feasible',
                'tagged_terminus', 'Mislocalized', 'is_Toxic_Gene']]

,GeneSymbol,Length,class,Pathogenic,Likely pathogenic,Uncertain significance,Benign,Likely benign,Conflicting classifications of pathogenicity,T_cell,HAP1,Main location,location_feasible,tagged_terminus,Mislocalized,is_Toxic_Gene
0,RAB32,225,cytoplasmic,0,0,26,0,1,0,Non-essential,non-essential,Cytosol,True,NaN,NaN,False
1,RIT2,217,cytoplasmic,0,0,17,1,0,0,Non-essential,non-essential,Nucleoplasm,True,NaN,NaN,False
2,SH3GL2,352,cytoplasmic,0,0,37,1,0,0,Non-essential,non-essential,Microtubules,True,NaN,NaN,False
3,SORL1,2214,secreted,19,7,568,90,256,22,Non-essential,non-essential,NaN,False,NaN,NaN,False
4,SV2C,727,cytoplasmic,0,0,101,8,13,0,Non-essential,non-essential,Nucleoplasm;Vesicles,True,NaN,0.0,False
5,TMEM175,504,cytoplasmic,0,0,95,4,7,0,Non-essential,non-essential,Nuclear membrane;Nucleoplasm,True,NaN,NaN,False
6,UNC13A,1703,cytoplasmic,6,4,256,27,116,5,Non-essential,non-essential,NaN,False,NaN,NaN,False


In [21]:
seven_genes_df.to_csv('MAVEable_seven_genes_20260917.csv', index=False)
print('Wrote MAVEable_seven_genes_20260917.csv')

# The uncollapsed, variant-level table is still in memory as CV_seven_hap1_loc_len
# if you want per-variant rows rather than the gene-level summary.
print(f'Variant-level rows available in CV_seven_hap1_loc_len: {len(CV_seven_hap1_loc_len):,}')

Wrote MAVEable_seven_genes_20260917.csv
Variant-level rows available in CV_seven_hap1_loc_len: 1,787
